# IPC2BNS-Verify — Phase 1: Deterministic Mapping & Query Normalization

This notebook demonstrates and verifies the Phase 1 components:
1. **Deterministic Concordance Lookup** (`lookup.py`): 100% exact, non-hallucinatory IPC ↔ BNS mapping.
2. **Ambiguity Handling Engine**: Explicit handling of repeals (§124A, §377, §497), splits (§33), and new offences (§111-113).
3. **Query Normalizer** (`normalizer.py`): Multi-tier extraction (Regex → Offence Ontology → LLM fallback).
4. **Unit Test Suite** (`test_concordance.py`): Full automated pytest verification.

---
## 1. Mount Google Drive & Environment Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/NLP_rspaper'
os.environ['IPC2BNS_PROJECT_ROOT'] = PROJECT_ROOT

if os.path.join(PROJECT_ROOT, 'code') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'code'))

print('Project root:', PROJECT_ROOT)
print('Environment configured successfully.')

Mounted at /content/drive
Project root: /content/drive/MyDrive/NLP_rspaper
Environment configured successfully.


---
## 2. Install Test Runner

In [2]:
!pip install -q pytest
print('Pytest ready.')

Pytest ready.


---
## 3. Initialize Deterministic Lookup Engine

In [3]:
from src.mapping.lookup import (
    get_lookup_engine,
    map_ipc_to_bns,
    map_bns_to_ipc,
    MappingStatus
)

engine = get_lookup_engine()
print(f'Concordance index loaded successfully from: {engine.concordance_path}')
print(f'Total indexed IPC sections: {len(engine.ipc_to_bns_index)}')
print(f'Total indexed BNS sections: {len(engine.bns_to_ipc_index)}')
print(f'Total valid BNS IDs: {len(engine.get_all_valid_bns_sections())}')

Concordance index loaded successfully from: /content/drive/MyDrive/NLP_rspaper/data/02_ground_truth/concordance_v1.csv
Total indexed IPC sections: 145
Total indexed BNS sections: 131
Total valid BNS IDs: 131


---
## 4. Interactive Lookup Demonstration (IPC ↔ BNS)

In [4]:
test_cases = ['302', '420', '375', '304B', '499', '503']

print('--- IPC -> BNS (Forward Lookup) ---')
for sec in test_cases:
    res = map_ipc_to_bns(sec)
    print(f'IPC §{res.query_section:4s} ({res.source_title[:28]:28s}) -> BNS §{res.target_section} [{res.status.value}]')

print('\n--- BNS -> IPC (Reverse Lookup) ---')
for sec in ['103', '318', '63', '80', '356', '351']:
    res = map_bns_to_ipc(sec)
    print(f'BNS §{res.query_section:4s} ({res.source_title[:28]:28s}) -> IPC §{res.target_section} [{res.status.value}]')

--- IPC -> BNS (Forward Lookup) ---
IPC §302  (Punishment for murder       ) -> BNS §103 [renumbered]
IPC §420  (Cheating and dishonestly ind) -> BNS §318 [renumbered]
IPC §375  (Rape                        ) -> BNS §63 [renumbered]
IPC §304B (Dowry death                 ) -> BNS §80 [renumbered]
IPC §499  (Defamation                  ) -> BNS §356 [renumbered]
IPC §503  (Criminal intimidation       ) -> BNS §351 [renumbered]

--- BNS -> IPC (Reverse Lookup) ---
BNS §103  (Punishment for murder       ) -> IPC §302 [renumbered]
BNS §318  (Cheating                    ) -> IPC §415 [merged]
BNS §63   (Rape                        ) -> IPC §375 [renumbered]
BNS §80   (Dowry death                 ) -> IPC §304B [renumbered]
BNS §356  (Defamation                  ) -> IPC §499 [merged]
BNS §351  (Criminal intimidation       ) -> IPC §503 [merged]


---
## 5. Critical Ambiguity & Edge Case Verification

Demonstrating the core novelty: **Ambiguous and non-1:1 provisions are NEVER hallucinated or silently force-mapped.**

In [5]:
edge_cases = [
    ('124A', 'Sedition (Repealed in BNS, narrower scope in §152)'),
    ('377',  'Unnatural Offences (Decriminalized / Struck down)'),
    ('497',  'Adultery (Struck down in Joseph Shine v UOI)'),
    ('33',   'Act / Omission (Split into BNS §2(1) and §2(25))'),
]

print('='*75)
print('AMBIGUITY & REPEAL VETO DEMONSTRATION')
print('='*75)

for sec, desc in edge_cases:
    res = map_ipc_to_bns(sec)
    print(f'\nQuery: IPC §{sec} ({desc})')
    print(f'  Target Section : {res.target_section}')
    print(f'  Status         : {res.status.value}')
    print(f'  Is Ambiguous   : {res.is_ambiguous}')
    print(f'  Matched List   : {res.all_matched_sections}')
    print(f'  Legal Notes    : {res.notes}')

AMBIGUITY & REPEAL VETO DEMONSTRATION

Query: IPC §124A (Sedition (Repealed in BNS, narrower scope in §152))
  Target Section : None
  Status         : repealed
  Is Ambiguous   : True
  Matched List   : []
  Legal Notes    : No direct BNS counterpart. BNS S.152 (acts endangering sovereignty/unity/integrity) is narrower in scope — flag as ambiguous DO NOT auto-map to 152

Query: IPC §377 (Unnatural Offences (Decriminalized / Struck down))
  Target Section : None
  Status         : repealed
  Is Ambiguous   : True
  Matched List   : []
  Legal Notes    : Decriminalized per Supreme Court Navtej Singh Johar v Union of India (2018); not carried to BNS

Query: IPC §497 (Adultery (Struck down in Joseph Shine v UOI))
  Target Section : None
  Status         : repealed
  Is Ambiguous   : True
  Matched List   : []
  Legal Notes    : Struck down by Supreme Court in Joseph Shine v Union of India (2018); not carried to BNS

Query: IPC §33 (Act / Omission (Split into BNS §2(1) and §2(25)))
  Targe

---
## 6. Query Normalization Layer (Free-Text → Canonical Section)

In [6]:
from src.mapping.normalizer import normalize_query

natural_queries = [
    'What is the new section for cheating in BNS 2023?',
    'punishment for murder under the new criminal code',
    'What happened to sedition under Section 124A of IPC?',
    'Where is dowry death penalized now?',
    'BNS section 103 provisions',
    'What is the law for defamation?',
]

print('='*75)
print('END-TO-END QUERY NORMALIZATION & LOOKUP PIPELINE')
print('='*75)

for q in natural_queries:
    norm = normalize_query(q)
    if norm.extracted_section:
        if norm.detected_act == 'BNS':
            mapping = map_bns_to_ipc(norm.extracted_section)
            dest_label = f'IPC §{mapping.target_section}'
        else:
            mapping = map_ipc_to_bns(norm.extracted_section)
            dest_label = f'BNS §{mapping.target_section}' if mapping.target_section else 'REPEALED / NO DIRECT 1:1 MAP'
        print(f'Query: "{q}"')
        print(f'  -> Extracted: {norm.detected_act} §{norm.extracted_section} (via {norm.method})')
        print(f'  -> Mapped To: {dest_label} [{mapping.status.value}]')
        print()
    else:
        print(f'Query: "{q}" -> Could not extract canonical section.')

END-TO-END QUERY NORMALIZATION & LOOKUP PIPELINE
Query: "What is the new section for cheating in BNS 2023?"
  -> Extracted: IPC §420 (via offence_lexicon)
  -> Mapped To: BNS §318 [renumbered]

Query: "punishment for murder under the new criminal code"
  -> Extracted: IPC §302 (via offence_lexicon)
  -> Mapped To: BNS §103 [renumbered]

Query: "What happened to sedition under Section 124A of IPC?"
  -> Extracted: IPC §124A (via regex)
  -> Mapped To: REPEALED / NO DIRECT 1:1 MAP [repealed]

Query: "Where is dowry death penalized now?"
  -> Extracted: IPC §304B (via offence_lexicon)
  -> Mapped To: BNS §80 [renumbered]

Query: "BNS section 103 provisions"
  -> Extracted: BNS §103 (via regex)
  -> Mapped To: IPC §302 [renumbered]

Query: "What is the law for defamation?"
  -> Extracted: IPC §499 (via offence_lexicon)
  -> Mapped To: BNS §356 [renumbered]



---
## 7. Run Full Automated Unit Test Suite (`test_concordance.py`)

In [7]:
test_file = os.path.join(PROJECT_ROOT, 'code/tests/test_concordance.py')
!python -m pytest "{test_file}" -v --color=yes

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.6.0, langsmith-0.11.1, anyio-4.14.2
collected 44 items                                                             

drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_concordance_table_loads_successfully PASSED [  2%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_concordance_schema_columns PASSED [  4%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[302-103] PASSED [  6%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[299-100] PASSED [  9%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[304A-106] PASSED [ 11%]
drive/MyDrive/NLP_rspaper/code/tests/test_concordance.py::test_deterministic_exact_mappings[30

---
## 8. Check Progress against WBS

In [8]:
!python "{PROJECT_ROOT}/check_progress.py" --root "{PROJECT_ROOT}" --write-report

# Project Progress Report
**Overall: 9/32 tasks complete (28%)**

_Generated: 2026-09-03T06:48:09_

## 0. Setup — 3/4 (75%)
- [x] Repo scaffolding + config system  `(code/src, code/configs)`
- [x] India Code raw text downloaded  `(data/00_raw/india_code)`
- [ ] Concordance source PDF(s) collected  `(data/00_raw/concordance_source_pdfs)`
- [x] Data Management Plan written  `(docs/IPC2BNS-Verify_Data_Management_Plan.md)`

## 1. Mapping Module — 5/5 (100%)
- [x] Ground-truth concordance table finalized  `(data/02_ground_truth/concordance_v1.csv)`
- [x] Concordance validation report reviewed  `(data/02_ground_truth/validation_report.csv)`
- [x] Deterministic lookup function implemented  `(code/src/mapping/lookup.py)`
- [x] Query normalizer implemented  `(code/src/mapping/normalizer.py)`
- [x] Mapping module unit tests  `(code/tests/test_concordance.py)`

## 2. Ingestion & Retrieval — 1/6 (17%)
- [ ] Section-level chunker implemented  `(code/src/ingestion/chunker.py)`
- [x] Cleaned section 